In [1]:
import pandas as pd
import pickle

# Load train_dataset 
train_data = pd.read_csv('datasets/train_dataset.csv')

# Separate features and target
X = train_data.drop(['isFraud'], axis=1)
y = train_data['isFraud']

# Load the 10-fold indices
with open("10_fold_indices.pkl", "rb") as f:
    splits = pickle.load(f)

print(f"Loaded {len(splits)} folds.")

Loaded 10 folds.


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import os

# -------------------- Identify Column Types --------------------
categorical_cols = X.select_dtypes(include="object").columns.tolist()
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Encode Categorical Columns
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le

# Standardize Numeric Columns
scaler = StandardScaler()
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

# -------------------- Dataset Class --------------------
class FraudDataset(Dataset):
    def __init__(self, X, y, cat_cols):
        self.X_num = X.drop(columns=cat_cols).values.astype(np.float32)
        self.X_cat = X[cat_cols].values.astype(np.int64)
        self.y = y.values.astype(np.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {
            'numerical': torch.tensor(self.X_num[idx]),
            'categorical': torch.tensor(self.X_cat[idx]),
            'label': torch.tensor(self.y[idx])
        }

# -------------------- Model Definition --------------------
class FraudDNN(nn.Module):
    def __init__(self, embedding_sizes, num_numeric):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(categories, min(50, (categories + 1) // 2))
            for categories in embedding_sizes
        ])
        self.embedding_dropout = nn.Dropout(0.3)

        total_embedding_dim = sum([emb.embedding_dim for emb in self.embeddings])

        self.fc = nn.Sequential(
            nn.Linear(total_embedding_dim + num_numeric, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )

    def forward(self, x_cat, x_num):
        x = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.cat(x, dim=1)
        x = self.embedding_dropout(x)
        x = torch.cat([x, x_num], dim=1)
        return self.fc(x)

# -------------------- Training Function --------------------
def train_model(model, train_loader, val_loader, device, patience=50, max_epochs=5000):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    best_auc = 0
    best_state = None
    rounds_without_improvement = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        for batch in train_loader:
            x_num = batch['numerical'].to(device)
            x_cat = batch['categorical'].to(device)
            y = batch['label'].to(device).unsqueeze(1)

            optimizer.zero_grad()
            out = model(x_cat, x_num)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

        # Validation
        model.eval()
        y_true, y_scores = [], []
        with torch.no_grad():
            for batch in val_loader:
                x_num = batch['numerical'].to(device)
                x_cat = batch['categorical'].to(device)
                y = batch['label'].to(device).unsqueeze(1)

                out = model(x_cat, x_num)
                y_true.extend(y.cpu().numpy())
                y_scores.extend(torch.sigmoid(out).cpu().numpy())

        auc = roc_auc_score(y_true, y_scores)
        print(f"Epoch {epoch} AUC-ROC: {auc:.4f}")

        if auc > best_auc:
            best_auc = auc
            best_state = model.state_dict()
            rounds_without_improvement = 0
        else:
            rounds_without_improvement += 1

        if rounds_without_improvement >= patience:
            print(f"Early stopping at epoch {epoch} | Best AUC: {best_auc:.4f}")
            break

    model.load_state_dict(best_state)
    return best_auc, y_true, y_scores

# -------------------- Run Cross-Validation --------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
auc_scores, accuracy_scores = [], []
precision_0, precision_1 = [], []
recall_0, recall_1 = [], []
f1_0, f1_1 = [], []

embedding_sizes = [X[col].nunique() for col in categorical_cols]
num_numeric = X.shape[1] - len(categorical_cols)

for fold, (train_idx, val_idx) in enumerate(splits, 1):
    print(f"\n Fold {fold}")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    train_ds = FraudDataset(X_train, y_train, categorical_cols)
    val_ds = FraudDataset(X_val, y_val, categorical_cols)

    train_loader = DataLoader(train_ds, batch_size=2048, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=2048, shuffle=False)

    model = FraudDNN(embedding_sizes, num_numeric).to(device)
    auc, y_true, y_scores = train_model(model, train_loader, val_loader, device)
    auc_scores.append(auc)

    y_pred = (np.array(y_scores) > 0.5).astype(int)
    y_true = np.array(y_true).astype(int)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average=None, zero_division=0)
    rec = recall_score(y_true, y_pred, average=None)
    f1 = f1_score(y_true, y_pred, average=None)

    accuracy_scores.append(acc)
    precision_0.append(prec[0])
    precision_1.append(prec[1])
    recall_0.append(rec[0])
    recall_1.append(rec[1])
    f1_0.append(f1[0])
    f1_1.append(f1[1])

    print(f" Fold {fold} AUC-ROC  : {auc:.4f}")
    print(f"   Accuracy     : {acc:.4f}")
    print(f"   Precision_0  : {prec[0]:.4f} | Precision_1 : {prec[1]:.4f}")
    print(f"   Recall_0     : {rec[0]:.4f} | Recall_1    : {rec[1]:.4f}")
    print(f"   F1 Score_0   : {f1[0]:.4f} | F1 Score_1  : {f1[1]:.4f}")

# -------------------- Summary --------------------
print("\n Cross-Validation Results (DNN with Embeddings)")
for i in range(len(auc_scores)):
    print(
        f"Fold {i+1}: AUC={auc_scores[i]:.4f} | Acc={accuracy_scores[i]:.4f} | "
        f"Prec_0={precision_0[i]:.4f} | Prec_1={precision_1[i]:.4f} | "
        f"Recall_0={recall_0[i]:.4f} | Recall_1={recall_1[i]:.4f} | "
        f"F1_0={f1_0[i]:.4f} | F1_1={f1_1[i]:.4f}"
    )

print("\n Final Averages:")
print(f"AUC-ROC     : {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
print(f"Accuracy    : {np.mean(accuracy_scores):.4f}")
print(f"Precision_0 : {np.mean(precision_0):.4f}")
print(f"Precision_1 : {np.mean(precision_1):.4f}")
print(f"Recall_0    : {np.mean(recall_0):.4f}")
print(f"Recall_1    : {np.mean(recall_1):.4f}")
print(f"F1 Score_0  : {np.mean(f1_0):.4f}")
print(f"F1 Score_1  : {np.mean(f1_1):.4f}")



 Fold 1
Epoch 1 AUC-ROC: 0.8712
Epoch 2 AUC-ROC: 0.8845
Epoch 3 AUC-ROC: 0.8951
Epoch 4 AUC-ROC: 0.8990
Epoch 5 AUC-ROC: 0.9010
Epoch 6 AUC-ROC: 0.9060
Epoch 7 AUC-ROC: 0.9095
Epoch 8 AUC-ROC: 0.9135
Epoch 9 AUC-ROC: 0.9159
Epoch 10 AUC-ROC: 0.9186
Epoch 11 AUC-ROC: 0.9195
Epoch 12 AUC-ROC: 0.9196
Epoch 13 AUC-ROC: 0.9230
Epoch 14 AUC-ROC: 0.9233
Epoch 15 AUC-ROC: 0.9240
Epoch 16 AUC-ROC: 0.9258
Epoch 17 AUC-ROC: 0.9276
Epoch 18 AUC-ROC: 0.9297
Epoch 19 AUC-ROC: 0.9305
Epoch 20 AUC-ROC: 0.9321
Epoch 21 AUC-ROC: 0.9322
Epoch 22 AUC-ROC: 0.9347
Epoch 23 AUC-ROC: 0.9326
Epoch 24 AUC-ROC: 0.9357
Epoch 25 AUC-ROC: 0.9332
Epoch 26 AUC-ROC: 0.9357
Epoch 27 AUC-ROC: 0.9365
Epoch 28 AUC-ROC: 0.9380
Epoch 29 AUC-ROC: 0.9368
Epoch 30 AUC-ROC: 0.9362
Epoch 31 AUC-ROC: 0.9394
Epoch 32 AUC-ROC: 0.9375
Epoch 33 AUC-ROC: 0.9381
Epoch 34 AUC-ROC: 0.9397
Epoch 35 AUC-ROC: 0.9385
Epoch 36 AUC-ROC: 0.9406
Epoch 37 AUC-ROC: 0.9414
Epoch 38 AUC-ROC: 0.9424
Epoch 39 AUC-ROC: 0.9416
Epoch 40 AUC-ROC: 0.9399
